# TikzTable: styling and drawing

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

This page continues from [TikzTable basics](tikztable.ipynb) with the features
that need the TikZ lattice: cell borders, gradient fills, and free-form drawing.

In [ ]:
import numpy as np
import pandas as pd

from gerrytools.latex import TikzTable

districts = pd.DataFrame(
    {
        "District": [f"CD {i}" for i in range(1, 9)],
        "BVAP share": [0.12, 0.18, 0.22, 0.31, 0.38, 0.44, 0.52, 0.58],
        "Dem share": [0.35, 0.41, 0.44, 0.47, 0.50, 0.55, 0.61, 0.66],
        "Polsby-Popper": [0.18, 0.22, 0.27, 0.31, 0.33, 0.35, 0.41, 0.44],
        "Pop. deviation": [0.004, 0.002, np.nan, 0.006, 0.001, 0.008, 0.003, 0.005],
    }
)
districts

## Cell borders

`set_cell_border(row, col, sides)` uses 1-based TikZ matrix coordinates, where row 1
is the first rendered row (the header) and column 1 is the leftmost column. Sides are
`"top"`, `"bottom"`, `"left"`, `"right"`, or `"all"`; adjacent bordered cells share
edges instead of double-drawing them.

In [ ]:
table = TikzTable(districts)
table.set_decimal_count(2)
table.set_cell_border(5, 2, "all")
table.set_cell_border(6, 2, "all")
table.highlight_rows(6, color="cc:lightblue!25!white")
print(table)

![Bordered cells over a highlighted row][tikztable-borders]

[tikztable-borders]: ../../_static/images/latex/tikztable-borders.png

## Gradient fills

The gradient formatters from the [TexTable page](textable.ipynb) work identically here.

In [ ]:
from gerrytools.latex.commands import tex_twocolor_gradient_command
from gerrytools.latex.formatters import (
    compose_formatters,
    diverging_gradient_formatter,
    round_decimals,
    wrap_with_tex_command,
)

table = TikzTable(districts)
table.set_decimal_count(2)
table.set_column_formatter(
    "Dem share",
    compose_formatters(
        diverging_gradient_formatter(
            lo=0.35,
            mid=0.50,
            hi=0.65,
            color_lo="cc:alizarin",
            color_mid="white",
            color_hi="cc:denim",
            command_name=None,
        ),
        round_decimals(2),
    ),
)
print(table)

![Diverging gradient in a TikzTable][tikztable-diverging]

[tikztable-diverging]: ../../_static/images/latex/tikztable-diverging.png

In [ ]:
table = TikzTable(districts)
table.set_nan_string("---")
table.set_number_formatter(compose_formatters(wrap_with_tex_command("heatmap"), round_decimals(2)))
table.document.add_command(tex_twocolor_gradient_command("heatmap"))
print(table)

![Two-color heatmap in a TikzTable][tikztable-heatmap]

[tikztable-heatmap]: ../../_static/images/latex/tikztable-heatmap.png

## Free-form drawing

`add_draw()` appends raw TikZ to the table's `\CodeAfter` block. Cells are addressable
as `(table-<row>-<col>)` with the usual TikZ anchors, and the row/column boundary
lattice as `(row-<i>)` / `(col-<j>)`, so a draw command can frame any block of cells.

In [ ]:
table = TikzTable(districts)
table.set_decimal_count(2)
table.add_draw(r"\draw[red, thick] (table-5-2.north west) rectangle (table-5-5.south east);")
print(table)

![A red frame drawn around a block of cells][tikztable-draws]

[tikztable-draws]: ../../_static/images/latex/tikztable-draws.png

## Related

- [TikzTable basics](tikztable.ipynb)
- [TexTable](textable.ipynb)
- [LaTeX API](../../api/latex.rst)